# **Gold Layer - Medication Availability by State**

## Description
This notebook creates a Gold-level aggregated dataset showing medication availability per U.S. state.

The dataset aggregates medication inventory across pharmacies to provide a state-level view of medication stock and pharmacy coverage.

It uses the unified Gold fact table that integrates diseases, medications, pharmacies, and inventory data.

## Source Table
- `capstone.gold.disease_medication_pharmacy_gold
`
## Target Table
- capstone.gold.medication_availability_state_gold

## Transformations
- Use the unified Gold fact table containing disease, medication, pharmacy, and inventory data
- Filter pharmacies where medication quantity is greater than zero
- Aggregate data by state and medication
- Calculate total available medication quantity
- Calculate pharmacy coverage per medication

## 1. Setup Environment

In [0]:
# Setup environment
# Use capstone catalog and ensure Gold schema exists

spark.sql("USE CATALOG capstone")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

## 2. Create Gold Table

In [0]:
%sql
CREATE OR REPLACE TABLE capstone.gold.medication_availability_state_gold AS

SELECT
    state,
    medication_id,
    medication_name,

    COUNT(DISTINCT pharmacy_id) AS pharmacy_count,
    SUM(quantity) AS total_quantity

FROM capstone.gold.disease_medication_pharmacy_gold

WHERE quantity > 0

GROUP BY
    state,
    medication_id,
    medication_name;

In [0]:
%sql
-- Preview the Gold dataset

SELECT *
FROM capstone.gold.medication_availability_state_gold
ORDER BY state, medication_name

## 3. Document Table **Columns**

In [0]:
%sql
-- Add column descriptions for Gold table

ALTER TABLE capstone.gold.medication_availability_state_gold
ALTER COLUMN state COMMENT 'State where the medication inventory is aggregated';

ALTER TABLE capstone.gold.medication_availability_state_gold
ALTER COLUMN medication_id COMMENT 'Unique identifier of the medication';

ALTER TABLE capstone.gold.medication_availability_state_gold
ALTER COLUMN medication_name COMMENT 'Standardized medication name';

ALTER TABLE capstone.gold.medication_availability_state_gold
ALTER COLUMN pharmacy_count COMMENT 'Number of pharmacies with available stock';

ALTER TABLE capstone.gold.medication_availability_state_gold
ALTER COLUMN total_quantity COMMENT 'Total medication quantity available across pharmacies';

## 4. Gold Data Quality Checks

In [0]:
%sql

-- Total number of rows in the Gold dataset
SELECT 
'Total Rows' AS check_type,
COUNT(*) AS result
FROM capstone.gold.medication_availability_state_gold

UNION ALL

-- Check for missing state values
SELECT
'Missing State',
COUNT(*)
FROM capstone.gold.medication_availability_state_gold
WHERE state IS NULL

UNION ALL

-- Check for missing medication identifiers
SELECT
'Missing Medication ID',
COUNT(*)
FROM capstone.gold.medication_availability_state_gold
WHERE medication_id IS NULL

UNION ALL

-- Check for invalid negative quantities
SELECT
'Negative Quantity',
COUNT(*)
FROM capstone.gold.medication_availability_state_gold
WHERE total_quantity < 0

UNION ALL

-- Check for duplicate business keys
SELECT
'Duplicate State + Medication',
COUNT(*)
FROM (
    SELECT state, medication_id
    FROM capstone.gold.medication_availability_state_gold
    GROUP BY state, medication_id
    HAVING COUNT(*) > 1
)

## 5. Validate Gold Table

In [0]:
%sql
-- Displays top medications by total available quantity
SELECT *
FROM capstone.gold.medication_availability_state_gold
ORDER BY total_quantity DESC
LIMIT 20

## 6. KPI - Top Medications by Availability
Identify medications with the highest total inventory across states.

In [0]:
%sql
-- Aggregates total quantity across all states

CREATE OR REPLACE VIEW capstone.kpi.kpi_top_medications AS
SELECT
medication_name,
SUM(total_quantity) AS total_stock

FROM capstone.gold.medication_availability_state_gold

GROUP BY medication_name

ORDER BY total_stock DESC

LIMIT 10

## 7. KPI - States with Highest Medication Availability
Identify states with the largest medication inventory.

In [0]:
%sql
-- Aggregates total medication stock per state

CREATE OR REPLACE VIEW capstone.kpi.kpi_states_highest_availability AS
SELECT
state,
SUM(total_quantity) AS total_stock

FROM capstone.gold.medication_availability_state_gold

GROUP BY state

ORDER BY total_stock DESC

LIMIT 10

## 8. KPI - States with Lowest Medication Availability
Identify states with limited medication availability.

In [0]:
%sql
-- Helps identify potential medication shortages

CREATE OR REPLACE VIEW capstone.kpi.kpi_states_lowest_availability AS
SELECT
state,
SUM(total_quantity) AS total_stock

FROM capstone.gold.medication_availability_state_gold

GROUP BY state

ORDER BY total_stock ASC

LIMIT 10

## 9. KPI - Pharmacy Coverage per Medication
Measure how widely medications are distributed across pharmacies.

In [0]:
%sql

-- Calculates average pharmacy presence per medication

CREATE OR REPLACE VIEW capstone.kpi.kpi_pharmacy_coverage_per_medication AS
SELECT
medication_name,
AVG(pharmacy_count) AS avg_pharmacy_coverage

FROM capstone.gold.medication_availability_state_gold

GROUP BY medication_name

ORDER BY avg_pharmacy_coverage DESC

LIMIT 50

## 10. KPI - Medication Variety per State
Measure how many different medications are available in each state.

In [0]:
%sql

-- Counts distinct medications available in each state
CREATE OR REPLACE VIEW capstone.kpi.kpi_medication_variety_by_state AS
SELECT
state,
COUNT(DISTINCT medication_id) AS medication_variety

FROM capstone.gold.medication_availability_state_gold

GROUP BY state

ORDER BY medication_variety DESC